In [ ]:
import pandas as pd
from pathlib import Path
import re
from plots import plot_histogram, plot_rank_boxplot
from evaluation import validate_distances

# Configuration
xp_name = "algorithm"
xp_base_path = Path("../experiments")

In [ ]:
# Discover experiment structure: {xp_base_path}/{xp_name}/{graph}/{query_hash}/{device_id}/*.csv
experiments = {}

if not xp_base_path.exists():
    raise ValueError(f"Experiment path does not exist: {xp_base_path}")

xp_name_path = xp_base_path / xp_name
if not xp_name_path.exists():
    raise ValueError(f"Experiment name path does not exist: {xp_name_path}")

# Scan through the hierarchy
for graph_dir in xp_name_path.iterdir():
    if not graph_dir.is_dir():
        continue
    graph_name = graph_dir.name
    
    for query_hash_dir in graph_dir.iterdir():
        if not query_hash_dir.is_dir():
            continue
        query_hash = query_hash_dir.name
        
        for device_id_dir in query_hash_dir.iterdir():
            if not device_id_dir.is_dir():
                continue
            device_id = device_id_dir.name
            
            # Create unique key for this experiment set
            exp_key = (graph_name, query_hash, device_id)
            
            if exp_key not in experiments:
                experiments[exp_key] = {
                    'dijkstra': [],
                    'deltastep': [],
                    'bellmanford': [],
                    'nearfar': []
                }
            
            # Find all CSV files in this directory
            for csv_file in device_id_dir.glob('*.csv'):
                # Parse filename: {timestamp}_{gitsha}_{variant}.csv
                match = re.match(r'(\d+)_([^_]+)_(.+)\.csv', csv_file.name)
                if not match:
                    continue
                
                timestamp = int(match.group(1))
                git_sha = match.group(2)
                variant = match.group(3)
                
                # Categorize by algorithm
                if variant == 'dijkstra':
                    experiments[exp_key]['dijkstra'].append((timestamp, csv_file))
                elif variant.startswith('deltastep_delta'):
                    experiments[exp_key]['deltastep'].append((timestamp, variant, csv_file))
                elif variant == 'bellmanford':
                    experiments[exp_key]['bellmanford'].append((timestamp, csv_file))
                elif variant.startswith('nearfar_delta'):
                    experiments[exp_key]['nearfar'].append((timestamp, variant, csv_file))

# Sort by timestamp (most recent first) and take the most recent for each algorithm
for exp_key in experiments:
    experiments[exp_key]['dijkstra'].sort(reverse=True, key=lambda x: x[0])
    experiments[exp_key]['deltastep'].sort(reverse=True, key=lambda x: x[0])
    experiments[exp_key]['bellmanford'].sort(reverse=True, key=lambda x: x[0])
    experiments[exp_key]['nearfar'].sort(reverse=True, key=lambda x: x[0])

print(f"Found {len(experiments)} experiment configurations:")
for exp_key, data in experiments.items():
    graph_name, query_hash, device_id = exp_key
    print(f"\nGraph: {graph_name}, Query: {query_hash}, Device: {device_id}")
    print(f"  Dijkstra: {len(data['dijkstra'])} files")
    print(f"  Deltastep: {len(data['deltastep'])} files")
    print(f"  BellmanFord: {len(data['bellmanford'])} files")
    print(f"  NearFar: {len(data['nearfar'])} files")

In [ ]:
# Select experiment configurations with all four algorithms present
complete_experiments = {}

for exp_key, data in experiments.items():
    has_dijkstra = len(data['dijkstra']) > 0
    has_deltastep = len(data['deltastep']) > 0
    has_bellmanford = len(data['bellmanford']) > 0
    has_nearfar = len(data['nearfar']) > 0
    
    if has_dijkstra and has_deltastep and has_bellmanford and has_nearfar:
        graph_name, query_hash, device_id = exp_key
        
        # Take the most recent file for each algorithm
        dijkstra_file = data['dijkstra'][0][1]
        deltastep_file = data['deltastep'][0][2]
        deltastep_variant = data['deltastep'][0][1]
        bellmanford_file = data['bellmanford'][0][1]
        nearfar_file = data['nearfar'][0][2]
        nearfar_variant = data['nearfar'][0][1]
        
        complete_experiments[exp_key] = {
            'dijkstra': dijkstra_file,
            'deltastep': deltastep_file,
            'deltastep_variant': deltastep_variant,
            'bellmanford': bellmanford_file,
            'nearfar': nearfar_file,
            'nearfar_variant': nearfar_variant
        }

if not complete_experiments:
    raise ValueError("No complete experiment sets found (need all 4 algorithms)")

print(f"\nFound {len(complete_experiments)} complete experiment set(s):")
for exp_key, files in complete_experiments.items():
    graph_name, query_hash, device_id = exp_key
    print(f"\nGraph: {graph_name}, Query: {query_hash}, Device: {device_id}")
    print(f"  Dijkstra: {files['dijkstra'].name}")
    print(f"  Deltastep ({files['deltastep_variant']}): {files['deltastep'].name}")
    print(f"  BellmanFord: {files['bellmanford'].name}")
    print(f"  NearFar ({files['nearfar_variant']}): {files['nearfar'].name}")

In [ ]:
# Load data for each complete experiment set
experiment_data = {}

for exp_key, files in complete_experiments.items():
    graph_name, query_hash, device_id = exp_key
    
    df_dijkstra = pd.read_csv(files['dijkstra'])
    df_deltastep = pd.read_csv(files['deltastep'])
    df_bellmanford = pd.read_csv(files['bellmanford'])
    df_nearfar = pd.read_csv(files['nearfar'])
    
    print(f"\nGraph: {graph_name}, Query: {query_hash}, Device: {device_id}")
    print(f"  Dijkstra: {len(df_dijkstra)} queries")
    print(f"  Deltastep: {len(df_deltastep)} queries")
    print(f"  BellmanFord: {len(df_bellmanford)} queries")
    print(f"  NearFar: {len(df_nearfar)} queries")
    
    # Merge all datasets
    df_merged = pd.merge(
        df_dijkstra,
        df_deltastep,
        on=['from_node_id', 'to_node_id', 'rank'],
        suffixes=('_dijkstra', '_deltastep')
    )
    
    df_merged = pd.merge(
        df_merged,
        df_bellmanford,
        on=['from_node_id', 'to_node_id', 'rank']
    )
    df_merged.rename(columns={'distance': 'distance_bellmanford', 'time': 'time_bellmanford'}, inplace=True)
    
    df_merged = pd.merge(
        df_merged,
        df_nearfar,
        on=['from_node_id', 'to_node_id', 'rank']
    )
    df_merged.rename(columns={'distance': 'distance_nearfar', 'time': 'time_nearfar'}, inplace=True)
    
    experiment_data[exp_key] = {
        'dijkstra': df_dijkstra,
        'deltastep': df_deltastep,
        'bellmanford': df_bellmanford,
        'nearfar': df_nearfar,
        'merged': df_merged
    }
    
    print(f"  Merged: {len(df_merged)} queries")

In [ ]:
# Check for distance errors for each experiment set
for exp_key, data in experiment_data.items():
    graph_name, query_hash, device_id = exp_key
    
    print(f"\n{'='*60}")
    print(f"Graph: {graph_name}, Query: {query_hash}, Device: {device_id}")
    print('='*60)
    
    algorithms = {
        'Dijkstra': data['dijkstra'],
        'Delta-stepping': data['deltastep'],
        'Bellman-Ford': data['bellmanford'],
        'Near-Far': data['nearfar']
    }
    
    validate_distances(algorithms)

In [ ]:
# Histogram comparing execution times for each experiment set
for exp_key, data in experiment_data.items():
    graph_name, query_hash, device_id = exp_key
    
    print(f"\n{'='*60}")
    print(f"Graph: {graph_name}, Query: {query_hash}, Device: {device_id}")
    print('='*60)
    
    algorithms = {
        'Dijkstra': data['dijkstra'],
        'Delta-stepping': data['deltastep'],
        'Bellman-Ford': data['bellmanford'],
        'Near-Far': data['nearfar']
    }
    
    plot_histogram(algorithms, f"{graph_name}_{device_id}")

In [ ]:
# Box plot by query rank for each experiment set
for exp_key, data in experiment_data.items():
    graph_name, query_hash, device_id = exp_key
    
    print(f"\n{'='*60}")
    print(f"Graph: {graph_name}, Query: {query_hash}, Device: {device_id}")
    print('='*60)
    
    algorithms = {
        'Dijkstra': data['dijkstra'],
        'Delta-stepping': data['deltastep'],
        'Bellman-Ford': data['bellmanford'],
        'Near-Far': data['nearfar']
    }
    
    plot_rank_boxplot(algorithms, f"{graph_name}_{device_id}")